# VectorDB Documents — Column Review

Review the structure and per-column content of:
- `VectorDB_Documents.json` (main collection)
- `VectorDBdev_Documents.json` (dev collection)

Both files are **MongoDB-export dumps**: a leading `[`, a trailing `]`, and pretty-printed JSON objects concatenated **without commas**. `json.load` does not accept this — the loader below uses `JSONDecoder.raw_decode` to stream-parse the records.

Each record has four top-level keys:
- `_id` — Mongo ObjectId wrapper (`{"$oid": "..."}`)
- `id` — integer primary key
- `vector_full_embeddings` — 1536-d float list
- `payload.metadata` — 19 metadata fields
- `payload.page_content` — text chunk

In [ ]:
import json
from json import JSONDecoder
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.max_colwidth", 140)

ROOT = Path("..")
MAIN_PATH = ROOT / "VectorDB_Documents.json"
DEV_PATH  = ROOT / "VectorDBdev_Documents.json"

def load_concatenated_json(path: Path):
    """Parse a MongoDB-style dump: optional [ ... ] wrapper with comma-less concatenated objects."""
    text = Path(path).read_text(encoding="utf-8").strip()
    if text.startswith("["):
        text = text[1:]
    if text.endswith("]"):
        text = text[:-1]
    dec = JSONDecoder()
    i, n = 0, len(text)
    out = []
    while i < n:
        while i < n and text[i] in " \n\r\t,":
            i += 1
        if i >= n:
            break
        obj, end = dec.raw_decode(text, i)
        out.append(obj)
        i = end
    return out

main_records = load_concatenated_json(MAIN_PATH)
dev_records  = load_concatenated_json(DEV_PATH)
print(f"main : {len(main_records):,} records  ({MAIN_PATH.stat().st_size/1e6:.1f} MB)")
print(f"dev  : {len(dev_records):,} records  ({DEV_PATH.stat().st_size/1e6:.1f} MB)")

## 1. Flatten to DataFrames

Explode `payload.metadata` into top-level columns, keep `page_content` as a column, and keep the raw embedding vector in a dedicated column.  A `split` column tags each row as `main` or `dev`.

In [ ]:
def records_to_df(records, split: str) -> pd.DataFrame:
    rows = []
    for r in records:
        payload  = r.get("payload", {}) or {}
        metadata = payload.get("metadata", {}) or {}
        row = {
            "split": split,
            "_id": (r.get("_id") or {}).get("$oid"),
            "id": r.get("id"),
            "embedding": r.get("vector_full_embeddings"),
            "page_content": payload.get("page_content"),
        }
        for k, v in metadata.items():
            row[f"meta.{k}"] = v
        rows.append(row)
    return pd.DataFrame(rows)

df_main = records_to_df(main_records, "main")
df_dev  = records_to_df(dev_records, "dev")
df      = pd.concat([df_main, df_dev], ignore_index=True)

print(f"Combined shape: {df.shape}")
print(f"\nColumns ({len(df.columns)}):")
for c in df.columns:
    print(f"  {c}")

## 2. Schema overview

Per column: dtype, non-null count, unique-value count, and one example value — separately for main vs dev so you can spot schema drift.

In [ ]:
def short_preview(val, maxlen=80):
    if val is None:
        return ""
    if isinstance(val, list):
        return f"list[{len(val)}]" + (f" e.g. {val[:2]}" if val else "")
    if isinstance(val, dict):
        return f"dict{list(val.keys())}"
    s = str(val).replace("\n", " ")
    return s if len(s) <= maxlen else s[:maxlen] + "…"

def column_summary(frame: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for c in frame.columns:
        s = frame[c]
        non_null = s.notna().sum()
        try:
            n_unique = s.nunique(dropna=True)
        except TypeError:  # unhashable (lists/dicts)
            n_unique = "—"
        sample = s.dropna().iloc[0] if non_null else None
        rows.append({
            "column": c,
            "dtype": str(s.dtype),
            "non_null": int(non_null),
            "null": int(len(s) - non_null),
            "n_unique": n_unique,
            "example": short_preview(sample),
        })
    return pd.DataFrame(rows).set_index("column")

print("── MAIN ──")
display(column_summary(df_main))
print("── DEV ──")
display(column_summary(df_dev))

## 3. ID and `_id` sanity

Check uniqueness of `id` within each split, and whether dev is a subset / disjoint set of main.

In [ ]:
main_ids = set(df_main["id"])
dev_ids  = set(df_dev["id"])
main_oids = set(df_main["_id"])
dev_oids  = set(df_dev["_id"])

print("── id ──")
print(f"main:  n={len(df_main):>4}  unique id={len(main_ids):>4}  "
      f"range=[{df_main['id'].min()}, {df_main['id'].max()}]")
print(f"dev :  n={len(df_dev):>4}  unique id={len(dev_ids):>4}  "
      f"range=[{df_dev['id'].min()}, {df_dev['id'].max()}]")
print(f"main ∩ dev ids : {len(main_ids & dev_ids)}")
print(f"dev ⊆ main     : {dev_ids.issubset(main_ids)}")
print(f"main ∪ dev ids : {len(main_ids | dev_ids)}")

print("\n── _id ($oid) ──")
print(f"main unique _id: {len(main_oids)}  | dev unique _id: {len(dev_oids)}")
print(f"main ∩ dev _id : {len(main_oids & dev_oids)}")

## 4. Embeddings

Dimensionality, any NaN/None entries, vector-norm distribution, and whether any two records are exact duplicates (suggests re-embedding of identical text).

In [ ]:
def embedding_stats(frame: pd.DataFrame, label: str):
    lens = frame["embedding"].apply(lambda v: len(v) if isinstance(v, list) else 0)
    any_missing = frame["embedding"].isna().sum()
    dims = lens.value_counts().to_dict()
    arr = np.asarray(frame["embedding"].tolist(), dtype=np.float32)
    norms = np.linalg.norm(arr, axis=1)
    nan_rows = int(np.isnan(arr).any(axis=1).sum())
    # Duplicate detection on first 32 dims (cheap proxy)
    sigs = [tuple(np.round(v[:32], 6)) for v in arr]
    dup_sigs = len(sigs) - len(set(sigs))
    print(f"── {label} ──")
    print(f"  records                : {len(frame):,}")
    print(f"  embedding length dist  : {dims}")
    print(f"  missing / None         : {any_missing}")
    print(f"  rows with NaN in vec   : {nan_rows}")
    print(f"  L2 norm  min/mean/max  : {norms.min():.6f} / {norms.mean():.6f} / {norms.max():.6f}")
    print(f"  likely duplicate vecs  : {dup_sigs} (matching first 32 dims)")
    return arr, norms

main_arr, main_norms = embedding_stats(df_main, "main")
print()
dev_arr,  dev_norms  = embedding_stats(df_dev, "dev")

all_norms = np.concatenate([main_norms, dev_norms])
lo, hi = float(all_norms.min()), float(all_norms.max())
if hi - lo < 1e-6:
    print(f"\nAll vectors are L2-normalised (norm ≈ {all_norms.mean():.6f}); "
          f"norm histogram skipped.")
else:
    pad = max((hi - lo) * 0.05, 1e-6)
    bins = np.linspace(lo - pad, hi + pad, 41)
    fig, ax = plt.subplots(figsize=(8, 3.5))
    ax.hist(main_norms, bins=bins, alpha=0.6, label="main", color="#4878d0")
    ax.hist(dev_norms,  bins=bins, alpha=0.6, label="dev",  color="#ee854a")
    ax.set_xlabel("L2 norm")
    ax.set_ylabel("records")
    ax.set_title("Embedding L2-norm distribution")
    ax.legend()
    plt.tight_layout()
    plt.show()

## 5. `page_content` — length & samples

In [ ]:
df["content_char_len"] = df["page_content"].fillna("").str.len()
df["content_word_len"] = df["page_content"].fillna("").str.split().str.len()

pcts = [0, 10, 25, 50, 75, 90, 95, 99, 100]
summary = (
    df.groupby("split")[["content_char_len", "content_word_len"]]
      .describe(percentiles=[p/100 for p in pcts if 0 < p < 100])
)
display(summary)

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
for ax, col, title in zip(axes, ["content_char_len", "content_word_len"],
                          ["Characters per chunk", "Words per chunk"]):
    for split, color in [("main", "#4878d0"), ("dev", "#ee854a")]:
        vals = df.loc[df["split"] == split, col]
        ax.hist(vals.clip(upper=vals.quantile(0.99)), bins=40,
                alpha=0.6, label=split, color=color)
    ax.set_xlabel(title + " (clipped p99)")
    ax.set_ylabel("records")
    ax.set_title(title)
    ax.legend()
plt.tight_layout()
plt.show()

print("── Shortest 3 chunks (main) ──")
for _, r in df_main.assign(l=df_main["page_content"].str.len()).nsmallest(3, "l").iterrows():
    print(f"  id={r['id']}  len={len(r['page_content'] or '')}  {r['page_content']!r}")
print("\n── Longest 3 chunks (main) — first 200 chars ──")
for _, r in df_main.assign(l=df_main["page_content"].str.len()).nlargest(3, "l").iterrows():
    txt = (r["page_content"] or "")[:200].replace("\n", " ")
    print(f"  id={r['id']}  len={len(r['page_content'] or '')}  {txt}…")

## 6. Metadata — per-column breakdown

Every `meta.*` field, value counts (top 10) and null rate. Low-cardinality fields show the full distribution; high-cardinality fields show only the most frequent values.

In [ ]:
meta_cols = [c for c in df.columns if c.startswith("meta.")]

def describe_metadata_column(col: str):
    s = df[col]
    # Normalize unhashables (lists/dicts) to JSON strings so value_counts works
    def _norm(v):
        if isinstance(v, (list, dict)):
            return json.dumps(v, ensure_ascii=False)[:120]
        return v
    s_norm = s.map(_norm)
    null_rate = s_norm.isna().mean()
    n_unique  = s_norm.nunique(dropna=True)
    print(f"\n── {col}   (null={null_rate:.1%}, unique={n_unique}) ──")
    if n_unique == 0:
        print("  (all null)")
        return
    vc = s_norm.value_counts(dropna=False).head(10)
    by_split = (
        df.assign(_v=s_norm)
          .groupby(["split", "_v"], dropna=False).size()
          .unstack("split", fill_value=0)
    )
    top = by_split.loc[vc.index] if set(vc.index).issubset(by_split.index) else by_split.head(10)
    display(top)

for c in meta_cols:
    describe_metadata_column(c)

## 7. Metadata cross-tabs

A few likely-interesting two-way breakdowns of low-cardinality fields.

In [ ]:
def safe_crosstab(a: str, b: str):
    if a not in df.columns or b not in df.columns:
        return
    print(f"\n── {a}  ×  {b} ──")
    display(pd.crosstab(df[a], df[b], margins=True, dropna=False))

for a, b in [
    ("meta.document_type",   "meta.document_status"),
    ("meta.document_type",   "meta.is_header"),
    ("meta.node_level",      "meta.is_header"),
    ("meta.has_requirements","meta.document_type"),
    ("split",                "meta.document_type"),
    ("split",                "meta.document_status"),
]:
    safe_crosstab(a, b)

## 8. `node_level` & `page_number` distributions

Histograms for the numeric metadata fields.

In [ ]:
numeric_meta = [
    c for c in ["meta.id", "meta.node_level", "meta.document_id", "meta.page_number"]
    if c in df.columns
]

n = len(numeric_meta)
if n:
    fig, axes = plt.subplots(1, n, figsize=(4.5 * n, 3.5))
    if n == 1:
        axes = [axes]
    for ax, col in zip(axes, numeric_meta):
        vals = pd.to_numeric(df[col], errors="coerce").dropna()
        ax.hist(vals, bins=30, color="#4878d0", edgecolor="white", linewidth=0.3)
        ax.set_title(col)
        ax.set_xlabel(col)
        ax.set_ylabel("records")
    plt.tight_layout()
    plt.show()

## 9. Record inspector

Set `INSPECT_SPLIT` and `INSPECT_ID` to view a single record (all fields, truncated embedding).

In [ ]:
INSPECT_SPLIT = "main"   # "main" or "dev"
INSPECT_ID    = None     # int id; leave None to pick the first record of the split

subset = df_main if INSPECT_SPLIT == "main" else df_dev
if INSPECT_ID is None:
    row = subset.iloc[0]
else:
    match = subset[subset["id"] == INSPECT_ID]
    row = match.iloc[0] if len(match) else None

if row is None:
    print(f"No record with id={INSPECT_ID} in {INSPECT_SPLIT}")
else:
    print(f"split={INSPECT_SPLIT}  id={row['id']}  _id={row['_id']}")
    emb = row["embedding"]
    print(f"embedding: dim={len(emb)}  first 5 = {emb[:5]}\n")
    print("── metadata ──")
    for c in [c for c in row.index if c.startswith("meta.")]:
        print(f"  {c:42s} {short_preview(row[c], 120)}")
    print("\n── page_content ──")
    print(row["page_content"])

## 10. Export a flat CSV view (optional)

Drops the 1536-d embedding and writes a human-reviewable CSV next to the source JSON files. Large-file storage rules still apply — if these outputs grow, move them into `output/` (the OneDrive junction).

In [ ]:
EXPORT = False   # set to True to write CSVs

if EXPORT:
    flat = df.drop(columns=["embedding"]).copy()
    # Stringify any list/dict metadata so CSV round-trips cleanly
    for c in flat.columns:
        if flat[c].map(lambda v: isinstance(v, (list, dict))).any():
            flat[c] = flat[c].map(lambda v: json.dumps(v, ensure_ascii=False) if isinstance(v, (list, dict)) else v)
    out_path = ROOT / "vectordb_documents_flat.csv"
    flat.to_csv(out_path, index=False, encoding="utf-8")
    print(f"Wrote {out_path}  ({out_path.stat().st_size/1e6:.2f} MB, {len(flat):,} rows)")